# Team Operations : Reset, Stop, Resume and Abort

In [1]:
import asyncio
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')
model_client = OpenAIChatCompletionClient(model='gpt-4o', api_key=api_key)

In [9]:
from autogen_agentchat.agents import AssistantAgent
add_1_agent_first = AssistantAgent(
    name = 'add_1_agent_first',
    model_client=model_client,
    system_message="Add 1 to the number, first number is 0. Give result as output, donn't give any explanations"
)

add_1_agent_second = AssistantAgent(
    name = 'add_1_agent_second',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)
 
add_1_agent_third = AssistantAgent(
    name = 'add_1_agent_third',
    model_client=model_client,
    system_message="Add 1 to the number. Give result as output."
)


In [10]:
from autogen_agentchat.teams import RoundRobinGroupChat

team = RoundRobinGroupChat(
    [add_1_agent_first, add_1_agent_second, add_1_agent_third],
    max_turns=3
)

In [11]:
from autogen_agentchat.ui import Console

await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3


TaskResult(messages=[TextMessage(id='266d4e9a-d05e-4239-b29b-4e5280c46a92', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=30, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 33, 756238, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='a529afaf-1b35-408d-8152-6172fb8ed79e', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 34, 494736, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='416a5a36-cddf-49e7-abd4-cc1db97b40f5', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 35, 136492, tzinfo=datetime.timezone.utc), content='3', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

# Resuming a Team
Teams are stateful and maintains the conversation history and context after each run, unless you reset the team.


We can resume a team to continue from where it left off by calling the run() or run_stream() method without a **new task**


In [12]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
4
---------- TextMessage (add_1_agent_second) ----------
5
---------- TextMessage (add_1_agent_third) ----------
6


TaskResult(messages=[TextMessage(id='cbb15377-cae9-49dd-aa57-7901388f3711', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=56, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 45, 714611, tzinfo=datetime.timezone.utc), content='4', type='TextMessage'), TextMessage(id='65364dce-f404-47ec-9a16-0e52433f8e57', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=55, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 46, 412435, tzinfo=datetime.timezone.utc), content='5', type='TextMessage'), TextMessage(id='565def35-52a6-4666-8d2d-7646868e4321', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=64, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 47, 81447, tzinfo=datetime.timezone.utc), content='6', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

In [13]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
7
---------- TextMessage (add_1_agent_second) ----------
8
---------- TextMessage (add_1_agent_third) ----------
9


TaskResult(messages=[TextMessage(id='13e87023-6ef4-45e9-8dcb-7a634aea4a67', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=82, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 53, 599883, tzinfo=datetime.timezone.utc), content='7', type='TextMessage'), TextMessage(id='58b9a85a-6d24-4af5-be83-c2c8174338ed', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=81, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 54, 202029, tzinfo=datetime.timezone.utc), content='8', type='TextMessage'), TextMessage(id='3753d8dc-cab6-41c4-bd52-7a6f2aed2136', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=89, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 10, 54, 771788, tzinfo=datetime.timezone.utc), content='9', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')

# team resumed from where it left off in the output above, and the first message is from the next agent after the last agent that spoke before the team stopped.

In [14]:
await Console(team.run_stream(task = 'What was the largest number you got in the result?'))

---------- TextMessage (user) ----------
What was the largest number you got in the result?
---------- TextMessage (add_1_agent_first) ----------
9
---------- TextMessage (add_1_agent_second) ----------
10
---------- TextMessage (add_1_agent_third) ----------
11


TaskResult(messages=[TextMessage(id='467afc1f-8c3c-458f-abc6-e54ebd7c2cc6', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 11, 9, 51227, tzinfo=datetime.timezone.utc), content='What was the largest number you got in the result?', type='TextMessage'), TextMessage(id='fc04216f-9e4a-488f-81dd-0ba62a9740e5', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=124, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 11, 9, 912508, tzinfo=datetime.timezone.utc), content='9', type='TextMessage'), TextMessage(id='953033ed-b592-4ace-bc8f-7d418b480b05', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=123, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 11, 10, 682618, tzinfo=datetime.timezone.utc), content='10', type='TextMessage'), TextMessage(id='f26aafd3-eafa-4496-af2a-9923ae1f938a', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=13

# Reset our Team

In [15]:
await team.reset() # on_reset() on all agents

In [16]:
await Console(team.run_stream())

---------- TextMessage (add_1_agent_first) ----------
1
---------- TextMessage (add_1_agent_second) ----------
2
---------- TextMessage (add_1_agent_third) ----------
3


TaskResult(messages=[TextMessage(id='8a92a3df-096c-4b67-b824-196002185396', source='add_1_agent_first', models_usage=RequestUsage(prompt_tokens=30, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 11, 59, 415516, tzinfo=datetime.timezone.utc), content='1', type='TextMessage'), TextMessage(id='cd8fe8b7-2843-40bb-8231-7277f52131de', source='add_1_agent_second', models_usage=RequestUsage(prompt_tokens=29, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 12, 0, 38513, tzinfo=datetime.timezone.utc), content='2', type='TextMessage'), TextMessage(id='d99a4cbc-4ee0-4040-885f-e351954be230', source='add_1_agent_third', models_usage=RequestUsage(prompt_tokens=39, completion_tokens=1), metadata={}, created_at=datetime.datetime(2026, 6, 27, 3, 12, 0, 648516, tzinfo=datetime.timezone.utc), content='3', type='TextMessage')], stop_reason='Maximum number of turns 3 reached.')